In [ ]:
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import os
import pandas as pd
from tensorflow.keras.models import load_model
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.regularizers import l2

In [ ]:
# Cargar imágenes sin binarizar, con división por clase
X_train_clean = np.load("data/X_train_class.npy")
X_test_clean = np.load("data/X_test_class.npy")

# Normalizar en [0,1] si es necesario (importante si no está hecho aún)
X_train_clean = X_train_clean.astype('float32') #/ 255.0
X_test_clean = X_test_clean.astype('float32') #/ 255.0

# Agregar canal si es necesario (verifica si tiene forma (64,128) o (64,128,1))
if len(X_train_clean.shape) == 3:
    X_train_clean = np.expand_dims(X_train_clean, axis=-1)
    X_test_clean = np.expand_dims(X_test_clean, axis=-1)

# Copia con ruido leve (por ejemplo el de 2p_10r)
X_train_noisy = np.load("data/train_ruido_2_10.npy")
X_test_noisy = np.load("data/test_ruido_2_10.npy")


In [ ]:
import numpy as np

X_train_noisy = np.load("data/train_ruido_2_10.npy")
X_test_noisy = np.load("data/test_ruido_2_10.npy")

print("Entrenamiento:", X_train_noisy.shape)
print("Prueba:", X_test_noisy.shape)
print("Valores:", np.unique(X_train_noisy))

In [ ]:
# Arquitectura del autoencoder con Dropout y L2
def construir_autoencoder_nb(input_shape=(64, 128, 1)):
    entrada = Input(shape=input_shape)

    # Codificador
    x = Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))(entrada)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    # Codificación (representación comprimida)
    x = Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))(x)

    # Decodificador
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)

    x = UpSampling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)

    salida = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

    modelo = Model(entrada, salida)
    return modelo

# Crear el modelo
autoencoder_nb = construir_autoencoder_nb()
autoencoder_nb.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])


In [ ]:
autoencoder_nb.summary()

In [ ]:
# Entrenamiento
history_nb = autoencoder_nb.fit(
    X_train_noisy, X_train_clean,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_noisy, X_test_clean),
    verbose=1
)


In [ ]:
# Pérdida
plt.figure(figsize=(10, 5))
plt.plot(history_nb.history['loss'], label='Loss (entrenamiento)')
plt.plot(history_nb.history['val_loss'], label='Val Loss (validación)', linestyle='--')
plt.title('Curva de pérdida - Modelo sin binarizar')
plt.xlabel('Épocas')
plt.ylabel('Pérdida (MSE)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# MAE
plt.figure(figsize=(10, 5))
plt.plot(history_nb.history['mae'], label='MAE (entrenamiento)', color='green', marker='o')
plt.plot(history_nb.history['val_mae'], label='Val MAE (validación)', linestyle='--', color='red')
plt.title('Curva de MAE - Modelo sin binarizar')
plt.xlabel('Épocas')
plt.ylabel('Error Absoluto Medio (MAE)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
decoded_imgs_nb = autoencoder_nb.predict(X_test_noisy)

In [ ]:
n = 7  # Número de imágenes a mostrar
plt.figure(figsize=(15, 5))

for i in range(n):
    # Imagen original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test_clean[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis('off')
        # Imagen ruidosa
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(X_test_noisy[i].squeeze(), cmap='gray')
    plt.title("Ruidosa")
    plt.axis('off')

    # Imagen reconstruida (ya reescalada)
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(decoded_imgs_nb[i].squeeze(), cmap='gray')
    plt.title("Reconstruida")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Número de imágenes a mostrar
n = 7
indices = np.random.choice(len(X_test_clean), n, replace=False)

# Umbral para binarizar
umbral = 0.5
decoded_binarized = (decoded_imgs_nb > umbral).astype(np.float32)

# Crear figura
plt.figure(figsize=(18, 8))

for i, idx in enumerate(indices):
    # Original
    ax = plt.subplot(4, n, i + 1)
    plt.imshow(X_test_clean[idx].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Original", fontsize=12)
    plt.axis('off')

    # Ruidosa
    ax = plt.subplot(4, n, i + 1 + n)
    plt.imshow(X_test_noisy[idx].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Ruidosa", fontsize=12)
    plt.axis('off')

    # Reconstruida
    ax = plt.subplot(4, n, i + 1 + 2*n)
    plt.imshow(decoded_imgs_nb[idx].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Reconstruida", fontsize=12)
    plt.axis('off')

    # Binarizada
    ax = plt.subplot(4, n, i + 1 + 3*n)
    plt.imshow(decoded_binarized[idx].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Binarizada", fontsize=12)
    plt.axis('off')

plt.suptitle("Comparación de imágenes: Original, Ruidosa, Reconstruida y Binarizada", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.losses import MeanSquaredError

# MAE manual
mae_nb = np.mean(np.abs(X_test_clean - decoded_imgs_nb))

# MSE (pérdida)
mse_fn = MeanSquaredError()
loss_mse_nb = mse_fn(X_test_clean, decoded_imgs_nb).numpy()

# PSNR promedio
psnr_values = [psnr(X_test_clean[i], decoded_imgs_nb[i], data_range=1.0) for i in range(len(X_test_clean))]
psnr_avg_nb = np.mean(psnr_values)

# SSIM promedio
ssim_values = [ssim(X_test_clean[i].squeeze(), decoded_imgs_nb[i].squeeze(), data_range=1.0) for i in range(len(X_test_clean))]
ssim_avg_nb = np.mean(ssim_values)

# Imprimir resultados
print(f"Loss (MSE): {loss_mse_nb:.4f}")
print(f"MAE: {mae_nb:.4f}")
print(f"PSNR (dB): {psnr_avg_nb:.2f}")
print(f"SSIM: {ssim_avg_nb:.4f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam

# Listas de combinaciones
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Directorios
ruido_dir = "data"
model_dir = "modelos_autoencoder_nb"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Diccionario para guardar métricas
mae_denoise = {}

# Entrenar con cada combinación de ruido
for n_patches in num_patches_list:
    for radius in radius_list:
        print(f"\nEntrenando con ruido: {n_patches} patches, radio {radius}")

        # Cargar datos ruidosos
        train_name = f"train_ruido_{n_patches}_{radius}.npy"
        test_name = f"test_ruido_{n_patches}_{radius}.npy"
        X_train_noisy = np.load(os.path.join(ruido_dir, train_name))
        X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

        # Clonar y compilar modelo pequeño (autoencoder_nb)
        modelo_denoise = clone_model(autoencoder_nb)
        modelo_denoise.compile(
            optimizer=Adam(),
            loss='mean_squared_error',
            metrics=['mae']
        )

        # Entrenar
        historial = modelo_denoise.fit(
            X_train_noisy, X_train_clean,
            epochs=50,
            batch_size=32,
            shuffle=True,
            validation_data=(X_test_noisy, X_test_clean),
            verbose=0
        )

        # Guardar modelo entrenado
        model_name = f"modelo_nb_{n_patches}_{radius}.keras"
        modelo_denoise.save(os.path.join(model_dir, model_name))

        # Guardar MAE de validación (última época)
        mae_val = historial.history['val_mae'][-1]
        mae_denoise[(n_patches, radius)] = mae_val
        print(f"MAE de validación: {mae_val:.4f}")

# Convertir resultados en DataFrame
df_mae = pd.DataFrame(
    index=sorted(set(k[0] for k in mae_denoise.keys())),   # número de parches
    columns=sorted(set(k[1] for k in mae_denoise.keys())), # radios
    dtype=float
)

for (n_patches, radius), mae in mae_denoise.items():
    df_mae.loc[n_patches, radius] = mae

# Dibujar heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_mae, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'MAE'})
plt.title("Heatmap del MAE por combinación de ruido (Modelo pequeño autoencoder_nb)")
plt.xlabel("Radio del parche")
plt.ylabel("Número de parches")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from tensorflow.keras.models import load_model
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# === Listas de combinaciones ===
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Directorios
ruido_dir = "data"
model_dir = "modelos_autoencoder_nb"

# Datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral para contar píxeles negros
umbral = 0.5

# Lista para guardar resultados
resultados = []

for n_patches in num_patches_list:
    for radius in radius_list:
        print(f"Procesando ruido estructurado: {n_patches} patches, radio {radius}")

        # Cargar modelo entrenado
        model_name = f"modelo_nb_{n_patches}_{radius}.keras"
        modelo_cargado = load_model(os.path.join(model_dir, model_name))

        # Cargar imágenes ruidosas
        test_name = f"test_ruido_{n_patches}_{radius}.npy"
        X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

        # Reconstrucciones
        decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

        # Calcular número de píxeles negros en cada imagen original
        num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1,2,3))

        # Calcular MSE por imagen
        mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

        # Guardar resultados
        for px, mse in zip(num_pixeles_negros, mse_values):
            resultados.append([px, mse, n_patches, radius])

# === Convertir a DataFrame ===
df = pd.DataFrame(resultados, columns=["pixeles_negros", "mse", "parches", "radio"])

# === Graficar: 1 subplot por número de parches ===
palette = sns.color_palette("mako", len(radius_list))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True, sharex=True)
axes = axes.flatten()

for i, n_patches in enumerate(num_patches_list):
    ax = axes[i]
    subset = df[df["parches"] == n_patches]
    sns.scatterplot(
        data=subset,
        x="pixeles_negros", y="mse",
        hue="radio", palette=palette,
        alpha=0.6, ax=ax
    )
    ax.set_title(f"{n_patches} parches")
    ax.set_xlabel("Número de píxeles negros")
    ax.set_ylabel("MSE")
    ax.legend(title="Radio")

plt.suptitle("Relación entre píxeles negros y error de reconstrucción (ruido estructurado)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from tensorflow.keras.models import load_model
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

# === Listas de combinaciones ===
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Directorios
ruido_dir = "data"
model_dir = "modelos_autoencoder_nb"

# Datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Umbral para contar píxeles negros
umbral = 0.5

# Lista para guardar resultados
resultados = []

for n_patches in num_patches_list:
    for radius in radius_list:
        print(f"Procesando ruido estructurado: {n_patches} patches, radio {radius}")

        # Cargar modelo entrenado
        model_name = f"modelo_nb_{n_patches}_{radius}.keras"
        modelo_cargado = load_model(os.path.join(model_dir, model_name))

        # Cargar imágenes ruidosas
        test_name = f"test_ruido_{n_patches}_{radius}.npy"
        X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

        # Reconstrucciones
        decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

        # Calcular número de píxeles negros en cada imagen original
        num_pixeles_negros = np.sum(X_test_clean < umbral, axis=(1,2,3))

        # Calcular MSE por imagen
        mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

        # Guardar resultados
        for px, mse in zip(num_pixeles_negros, mse_values):
            resultados.append([px, mse, n_patches, radius])

# === Convertir a DataFrame ===
df = pd.DataFrame(resultados, columns=["pixeles_negros", "mse", "parches", "radio"])

# === Graficar: 1 subplot por número de parches ===
palette = sns.color_palette("mako", len(radius_list))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True, sharex=True)
axes = axes.flatten()

for i, n_patches in enumerate(num_patches_list):
    ax = axes[i]
    subset = df[df["parches"] == n_patches]

    # Scatter
    sns.scatterplot(
        data=subset, x="pixeles_negros", y="mse",
        hue="radio", palette=palette, alpha=0.6, ax=ax
    )

    # Agregar regresiones y correlaciones
    for j, radius in enumerate(radius_list):
        sub_sub = subset[subset["radio"] == radius]
        if len(sub_sub) > 1:
            # Línea de regresión
            sns.regplot(
                data=sub_sub, x="pixeles_negros", y="mse",
                scatter=False, ci=None, ax=ax,
                line_kws={"color": palette[j]}
            )
            # Correlación de Spearman
            rho, _ = spearmanr(sub_sub["pixeles_negros"], sub_sub["mse"])
            ax.text(
                0.95, 0.90 - j*0.07,
                f"ρ({radius}) = {rho:.2f}",
                transform=ax.transAxes, ha="right", color="black"
            )

    ax.set_title(f"{n_patches} parches")
    ax.set_xlabel("Número de píxeles negros")
    ax.set_ylabel("MSE")

    if i == 0:
        ax.legend(title="Radio")
    else:
        ax.get_legend().remove()

plt.suptitle("Relación entre píxeles negros y error de reconstrucción (ruido estructurado)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model, clone_model
from tensorflow.keras.optimizers import Adam

# === Configuración ===
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

ruido_dir = "data"
model_dir = "modelos_autoencoder_nb"

# Cargar datos limpios
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Lista de resultados
resultados = []

# Procesar cada combinación de ruido estructurado
for n_patches in num_patches_list:
    for radius in radius_list:
        key = f"{n_patches}p_{radius}r"
        print(f"Procesando combinación: {key}")

        # Cargar modelo
        model_name = f"modelo_nb_{n_patches}_{radius}.keras"
        modelo_cargado = load_model(os.path.join(model_dir, model_name))

        # Cargar datos ruidosos
        test_name = f"test_ruido_{n_patches}_{radius}.npy"
        X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

        # Reconstrucciones
        decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

        # Calcular MSE por imagen
        mse_values = np.mean((X_test_clean - decoded_imgs) ** 2, axis=(1,2,3))

        # Guardar resultados en lista
        for mse in mse_values:
            resultados.append([key, mse])

# Convertir a DataFrame
df_estruct = pd.DataFrame(resultados, columns=["config_ruido", "mse"])

# === Graficar Boxplot ===
plt.figure(figsize=(10,6))
sns.boxplot(data=df_estruct, x="config_ruido", y="mse", palette="mako")
plt.xticks(rotation=45)
plt.xlabel("Configuración de ruido estructurado (parches_radio)")
plt.ylabel("MSE por imagen")
plt.title("Distribución de errores de reconstrucción (ruido estructurado)")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === Parámetros ===
indices = [5, 25, 60]   # Usa valores menores que len(X_test_clean)
n_patches, radius = 3, 20

# Cargar imágenes limpias y ruidosas
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))
X_test_noisy = np.load(os.path.join(ruido_dir, f"test_ruido_{n_patches}_{radius}.npy"))

# Reconstrucciones del modelo entrenado
model_name = f"modelo_nb_{n_patches}_{radius}.keras"
modelo_cargado = load_model(os.path.join(model_dir, model_name))
decoded_imgs = modelo_cargado.predict(X_test_noisy, verbose=0)

# === Visualización ===
fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3*len(indices)))

for i, idx in enumerate(indices):
    original = X_test_clean[idx].squeeze()
    noisy = X_test_noisy[idx].squeeze()
    reconstruida = decoded_imgs[idx].squeeze()
    error_map = np.abs(original - reconstruida)

    axes[i,0].imshow(original, cmap="gray")
    axes[i,0].set_title("Original")
    axes[i,0].axis("off")

    axes[i,1].imshow(noisy, cmap="gray")
    axes[i,1].set_title(f"Noisy ({n_patches}p, r={radius})")
    axes[i,1].axis("off")

    axes[i,2].imshow(reconstruida, cmap="gray")
    axes[i,2].set_title("Reconstruida")
    axes[i,2].axis("off")

    im = axes[i,3].imshow(error_map, cmap="plasma", vmin=0, vmax=1)
    axes[i,3].set_title("Mapa de error")
    axes[i,3].axis("off")
    fig.colorbar(im, ax=axes[i,3])

plt.suptitle("Heatmaps de error por posición (ruido estructurado)", fontsize=14)
plt.tight_layout()
plt.show()


### Modelo con más filtros o capas

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.regularizers import L2

# Arquitectura del autoencoder con más capas y filtros
def construir_autoencoder_profundo(input_shape=(64, 128, 1)):
    entrada = Input(shape=input_shape)

    # Codificador
    x = Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(entrada)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Dropout(0.3)(x)

    # Codificación (cuello de botella)
    x = Conv2D(512, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)

    # Decodificador
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
    x = Dropout(0.3)(x)

    x = UpSampling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
    x = Dropout(0.3)(x)

    x = UpSampling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
    x = Dropout(0.3)(x)

    # Capa de salida
    salida = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

    modelo = Model(entrada, salida)
    return modelo

# Crear y compilar el modelo
autoencoder_profundo = construir_autoencoder_profundo()
autoencoder_profundo.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])


In [ ]:
autoencoder_profundo.summary()

In [ ]:
# Entrenamiento
history_prof = autoencoder_profundo.fit(
    X_train_noisy, X_train_clean,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_data=(X_test_noisy, X_test_clean),
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Pérdida
plt.figure(figsize=(10, 5))
plt.plot(history_prof.history['loss'], label='Loss (entrenamiento)', color='blue')
plt.plot(history_prof.history['val_loss'], label='Val Loss (validación)', color='orange', linestyle='--')
plt.title("Curva de pérdida - Modelo Profundo")
plt.xlabel("Épocas")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.show()

# MAE
plt.figure(figsize=(10, 5))
plt.plot(history_prof.history['mae'], label='MAE (entrenamiento)', color='green', marker='o')
plt.plot(history_prof.history['val_mae'], label='Val MAE (validación)', color='red', linestyle='--')
plt.title("Curva de MAE - Modelo Profundo")
plt.xlabel("Épocas")
plt.ylabel("Error Absoluto Medio (MAE)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Reconstrucción
decoded_imgs_profundo = autoencoder_profundo.predict(X_test_noisy)

# Mostrar imágenes
n = 5
plt.figure(figsize=(15, 5))
for i in range(n):
    # Ruidosa
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test_noisy[i].squeeze(), cmap='gray')
    plt.title("Ruidosa")
    plt.axis('off')

    # Original
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(X_test_clean[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis('off')

    # Reconstruida
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(decoded_imgs_profundo[i].squeeze(), cmap='gray')
    plt.title("Reconstruida")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# MAE manual
mae_p = np.mean(np.abs(X_test_clean - decoded_imgs_profundo))

# MSE
mse_fn = MeanSquaredError()
mse_p = mse_fn(X_test_clean, decoded_imgs_profundo).numpy()

# PSNR
psnr_values = [psnr(X_test_clean[i], decoded_imgs_profundo[i], data_range=1.0) for i in range(len(X_test_clean))]
psnr_avg = np.mean(psnr_values)

# SSIM
ssim_values = [ssim(X_test_clean[i].squeeze(), decoded_imgs_profundo[i].squeeze(), data_range=1.0) for i in range(len(X_test_clean))]
ssim_avg = np.mean(ssim_values)

# Mostrar
print(f"Loss (MSE): {mse_p:.4f}")
print(f"MAE: {mae_p:.4f}")
print(f"PSNR (dB): {psnr_avg:.2f}")
print(f"SSIM: {ssim_avg:.4f}")


### Lo corro para todas las combinaciones

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam

# Listas de combinaciones
num_patches_list = [2, 3, 4, 5]
radius_list = [10, 20, 30]

# Directorios
ruido_dir = "data"
model_dir = "modelos_autoencoder_profundo"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Diccionario para guardar métricas
mae_denoise = {}

# Entrenar con cada combinación de ruido
for n_patches in num_patches_list:
    for radius in radius_list:
        print(f"\nEntrenando con ruido: {n_patches} patches, radio {radius}")

        # Cargar datos ruidosos
        train_name = f"train_ruido_{n_patches}_{radius}.npy"
        test_name = f"test_ruido_{n_patches}_{radius}.npy"
        X_train_noisy = np.load(os.path.join(ruido_dir, train_name))
        X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

        # Clonar y compilar modelo profundo
        modelo_denoise = clone_model(autoencoder_profundo)
        modelo_denoise.compile(
            optimizer=Adam(),
            loss='mean_squared_error',
            metrics=['mae']
        )

        # Entrenar
        historial = modelo_denoise.fit(
            X_train_noisy, X_train_clean,
            epochs=50,
            batch_size=32,
            shuffle=True,
            validation_data=(X_test_noisy, X_test_clean),
            verbose=0
        )

        # Guardar modelo entrenado
        model_name = f"modelo_profundo_{n_patches}_{radius}.keras"
        modelo_denoise.save(os.path.join(model_dir, model_name))

        # Guardar MAE de validación (última época)
        mae_val = historial.history['val_mae'][-1]
        mae_denoise[(n_patches, radius)] = mae_val
        print(f"MAE de validación: {mae_val:.4f}")

# Convertir resultados en DataFrame
df_mae = pd.DataFrame(
    index=sorted(set(k[0] for k in mae_denoise.keys())),   # número de parches
    columns=sorted(set(k[1] for k in mae_denoise.keys())), # radios
    dtype=float
)

for (n_patches, radius), mae in mae_denoise.items():
    df_mae.loc[n_patches, radius] = mae

# Dibujar heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_mae, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'MAE'})
plt.title("Heatmap del MAE por combinación de ruido (Modelo profundo)")
plt.xlabel("Radio del parche")
plt.ylabel("Número de parches")
plt.tight_layout()
plt.show()
